# SoundScout — Music Recommendation System

A portfolio-ready music recommendation prototype built with **Python, scikit-learn, and Gradio**. Run this notebook in Google Colab to upload the Spotify tracks CSV, clean data, explore audio features, train an audio-based and hybrid recommender, evaluate recommendations, and launch a shareable `gradio.live` demo.

## Features
- Search tracks by **song title** or **artist**.
- Recommend similar songs using standardized audio features and cosine similarity.
- Compare **Audio-only** against **Hybrid** ranking that also incorporates genre overlap.
- Review EDA charts and offline evaluation outputs.

> The dataset contains track metadata and audio features, but no lyrics. This version intentionally does not include lyric search. The public demo URL is temporary and works while the Colab runtime is active.

## 1. Install dependencies
Run the cells in order. No CSV is included in the repository; upload your own copy of the dataset in the next step.

In [ ]:
%pip -q install -U gradio pandas numpy scikit-learn matplotlib seaborn

## 2. Upload the Spotify Tracks CSV

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Please upload the Spotify tracks dataset as a CSV file.")

file_path = csv_files[0]
raw_df = pd.read_csv(file_path)

print(f"Loaded: {file_path}")
print(f"Rows: {len(raw_df):,} | Columns: {len(raw_df.columns)}")
display(raw_df.head())
print("Columns:", raw_df.columns.tolist())

## 3. Clean and prepare the data
Repeated `track_id` records may represent the same Spotify track assigned to multiple genres. The recommender keeps one row per track but retains all genre labels in `all_genres`. The original CSV is not modified.

In [ ]:
import numpy as np

required_columns = {
    "track_id", "track_name", "artists", "track_genre", "popularity",
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo"
}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

clean_df = raw_df.dropna(subset=["track_id", "track_name", "artists"]).copy()
clean_df["track_id"] = clean_df["track_id"].astype(str)

genre_map = (
    clean_df.groupby("track_id")["track_genre"]
    .agg(lambda s: sorted({str(v) for v in s.dropna()}))
    .rename("all_genres")
)

# Preserve one row per track_id and all associated genre labels.
df = clean_df.drop_duplicates(subset="track_id", keep="first").copy()
df = df.merge(genre_map, on="track_id", how="left", validate="one_to_one")
df["duration_min"] = df["duration_ms"] / 60000 if "duration_ms" in df.columns else np.nan

audio_features = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo"
]
for col in audio_features + ["popularity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

model_df = df.dropna(subset=audio_features).reset_index(drop=True).copy()
print("DATA CLEANING SUMMARY")
print(f"Original rows: {len(raw_df):,}")
print(f"Rows with essential track identity: {len(clean_df):,}")
print(f"Unique tracks: {len(df):,}")
print(f"Tracks usable for recommendation: {len(model_df):,}")
print(f"Unique source genre labels: {clean_df['track_genre'].nunique():,}")
print(f"Missing values in model features: {model_df[audio_features].isna().sum().sum()}")
display(model_df[["track_name", "artists", "track_genre", "all_genres", "popularity"]].head())

## 4. Exploratory data analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(model_df["popularity"].dropna(), bins=30, ax=axes[0, 0])
axes[0, 0].set_title("Track Popularity Distribution")
axes[0, 0].set_xlabel("Popularity")

if "duration_min" in model_df.columns:
    duration_data = model_df.loc[model_df["duration_min"].between(0, 15), "duration_min"]
    sns.histplot(duration_data, bins=40, ax=axes[0, 1])
    axes[0, 1].set_title("Track Duration (up to 15 minutes)")
    axes[0, 1].set_xlabel("Duration (minutes)")

model_df[["danceability", "energy", "valence"]].plot(
    kind="hist", bins=30, alpha=0.55, ax=axes[1, 0]
)
axes[1, 0].set_title("Audio Feature Distributions")

model_df["track_genre"].value_counts().head(10).sort_values().plot(
    kind="barh", ax=axes[1, 1]
)
axes[1, 1].set_title("Top 10 Genre Labels by Row Count")
axes[1, 1].set_xlabel("Rows")

plt.tight_layout()
plt.show()

print("EDA summary")
print(f"Unique tracks: {model_df['track_id'].nunique():,}")
print(f"Source genre labels: {model_df['track_genre'].nunique():,}")
print(f"Mean popularity: {model_df['popularity'].mean():.2f}")
print(f"Mean danceability: {model_df['danceability'].mean():.3f}")
print(f"Mean energy: {model_df['energy'].mean():.3f}")
print(f"Mean valence: {model_df['valence'].mean():.3f}")

corr_features = ["popularity", *audio_features]
corr = model_df[corr_features].corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Matrix: Audio Features and Popularity")
plt.tight_layout()
plt.show()

## 5. Train the recommendation engine
All nine audio features are standardized before nearest-neighbor retrieval. Cosine distance is used to retrieve similar tracks. Hybrid ranking combines audio similarity and Jaccard similarity between each pair of genre-label sets.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

scaler = StandardScaler()
X_scaled = scaler.fit_transform(model_df[audio_features])

nn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)
nn_model.fit(X_scaled)

print("Recommendation engine ready")
print(f"Tracks: {len(model_df):,}")
print(f"Audio features: {len(audio_features)}")
print("Algorithm: nearest neighbors with cosine distance")

## 6. Quick recommendation test

In [ ]:
def recommend_songs(song_name, n_recommendations=10):
    query = str(song_name).strip().casefold()
    matches = model_df[model_df["track_name"].astype(str).str.casefold() == query]
    if matches.empty:
        print(f"Song '{song_name}' was not found. Try a different title.")
        return pd.DataFrame()

    source_pos = int(matches.index[0])
    n_neighbors = min(int(n_recommendations) + 1, len(model_df))
    distances, indices = nn_model.kneighbors(
        X_scaled[source_pos].reshape(1, -1), n_neighbors=n_neighbors
    )
    rows = []
    for distance, idx in zip(distances[0], indices[0]):
        if int(idx) == source_pos:
            continue
        row = model_df.iloc[int(idx)]
        rows.append({
            "track_name": row["track_name"],
            "artists": row["artists"],
            "track_genre": row["track_genre"],
            "popularity": row["popularity"],
            "audio_similarity": round(float(1 - distance), 4)
        })
        if len(rows) >= n_recommendations:
            break
    return pd.DataFrame(rows)

# Change this title to any title found in the catalogue.
display(recommend_songs("Comedy", 10))

## 7. Offline evaluation on 100 query tracks
Genre Match@10 is a proxy metric, not user satisfaction. Because Hybrid explicitly uses genre overlap, genre-based evaluation naturally favors that method. Additional metrics track artist variety, audio diversity, popularity, and catalogue coverage.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import os

K = 10
CANDIDATE_POOL = 100
AUDIO_WEIGHT = 0.75
rng = np.random.default_rng(42)
sample_positions = rng.choice(len(model_df), size=min(100, len(model_df)), replace=False)


def genres_for_row(row):
    value = row.get("all_genres", [])
    return set(value) if isinstance(value, (list, tuple, set)) else set()


def evaluate_query(source_pos):
    source = model_df.iloc[source_pos]
    source_genres = genres_for_row(source)
    n_candidates = min(CANDIDATE_POOL + 1, len(model_df))
    distances, indices = nn_model.kneighbors(
        X_scaled[source_pos].reshape(1, -1), n_neighbors=n_candidates
    )
    candidates = []
    for distance, idx in zip(distances[0], indices[0]):
        idx = int(idx)
        if idx == source_pos:
            continue
        row = model_df.iloc[idx]
        candidate_genres = genres_for_row(row)
        union = source_genres | candidate_genres
        inter = source_genres & candidate_genres
        audio_sim = float(1 - distance)
        genre_sim = len(inter) / len(union) if union else 0.0
        hybrid_score = AUDIO_WEIGHT * audio_sim + (1 - AUDIO_WEIGHT) * genre_sim
        candidates.append({
            "position": idx,
            "audio_similarity": audio_sim,
            "hybrid_score": hybrid_score,
            "genres": candidate_genres
        })

    rankings = {
        "Audio-only": sorted(candidates, key=lambda x: x["audio_similarity"], reverse=True)[:K],
        "Hybrid": sorted(candidates, key=lambda x: x["hybrid_score"], reverse=True)[:K]
    }
    out = []
    for model_name, ranking in rankings.items():
        positions = [x["position"] for x in ranking]
        recs = model_df.iloc[positions]
        genre_match = np.mean([
            bool(source_genres & genres_for_row(model_df.iloc[pos])) for pos in positions
        ]) if positions else np.nan
        artist_diversity = recs["artists"].nunique() / len(recs) if len(recs) else np.nan
        if len(positions) > 1:
            pairwise = cosine_similarity(X_scaled[positions])
            pairwise_sim = pairwise[np.triu_indices(len(positions), k=1)]
            intra_list_diversity = float(np.mean(1 - pairwise_sim))
        else:
            intra_list_diversity = np.nan
        out.append({
            "test_song": source["track_name"],
            "test_artist": source["artists"],
            "model": model_name,
            "genre_match@10": genre_match,
            "artist_diversity@10": artist_diversity,
            "intra_list_diversity": intra_list_diversity,
            "mean_recommendation_popularity": recs["popularity"].mean(),
            "recommended_track_ids": recs["track_id"].astype(str).tolist()
        })
    return out

records = []
for pos in sample_positions:
    records.extend(evaluate_query(int(pos)))

evaluation_detail = pd.DataFrame(records)
metric_cols = [
    "genre_match@10", "artist_diversity@10", "intra_list_diversity",
    "mean_recommendation_popularity"
]
evaluation_summary = evaluation_detail.groupby("model")[metric_cols].mean()
coverage = {}
for name in ["Audio-only", "Hybrid"]:
    sub = evaluation_detail[evaluation_detail["model"] == name]
    unique_ids = {tid for items in sub["recommended_track_ids"] for tid in items}
    coverage[name] = len(unique_ids) / len(model_df)
evaluation_summary["catalog_coverage"] = pd.Series(coverage)

display(evaluation_summary.round(3))

os.makedirs("outputs", exist_ok=True)
evaluation_summary.to_csv("outputs/evaluation_summary.csv")
evaluation_detail.drop(columns=["recommended_track_ids"]).to_csv(
    "outputs/evaluation_100_detailed.csv", index=False
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
plot_metrics = [
    ("genre_match@10", "Genre Match@10"),
    ("artist_diversity@10", "Artist Diversity@10"),
    ("intra_list_diversity", "Intra-list Diversity"),
    ("mean_recommendation_popularity", "Mean Recommendation Popularity")
]
for ax, (metric, title) in zip(axes.flat, plot_metrics):
    evaluation_summary[metric].plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Average score")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig("outputs/recommendation_evaluation.png", dpi=200, bbox_inches="tight")
plt.show()
print("Evaluation files saved in outputs/.")

## 8. Launch the Gradio app
The demo searches by artist or title. It does not search lyrics because the selected dataset does not contain lyric text.

In [ ]:
import gradio as gr

app_df = model_df.copy().reset_index(drop=True)
X_app = np.asarray(X_scaled)


def _genres(row):
    value = row.get("all_genres", [])
    if isinstance(value, (list, tuple, set)):
        return {str(g) for g in value if pd.notna(g)}
    genre = row.get("track_genre", None)
    return {str(genre)} if pd.notna(genre) else set()


def search_music(search_type, query):
    empty = pd.DataFrame(columns=["track_name", "artists", "track_genre", "popularity"])
    if not query or not query.strip():
        return gr.update(choices=[], value=None), empty, "Enter a search term."
    q = query.strip()
    column = "track_name" if search_type == "Song title" else "artists"
    found = app_df[app_df[column].astype(str).str.contains(q, case=False, na=False, regex=False)].copy()
    if found.empty:
        return gr.update(choices=[], value=None), empty, f"No results for '{q}'."
    found = found.sort_values("popularity", ascending=False).drop_duplicates("track_id").head(100)
    table = found[["track_name", "artists", "track_genre", "popularity"]].reset_index(drop=True)
    choices = []
    for _, row in found.iterrows():
        label = f"{row['track_name']} — {row['artists']} [{row['track_genre']}]"
        choices.append((label, str(row["track_id"])))
    return gr.update(choices=choices, value=choices[0][1]), table, f"Found {len(found)} tracks. Select one and recommend songs."


def recommend_music(track_id, model_type, top_k, audio_weight):
    columns = ["track_name", "artists", "track_genre", "popularity", "audio_similarity", "genre_similarity", "recommendation_score"]
    empty = pd.DataFrame(columns=columns)
    if not track_id:
        return empty, "Search for a track and select it first."
    positions = np.flatnonzero(app_df["track_id"].astype(str).to_numpy() == str(track_id))
    if len(positions) == 0:
        return empty, "The selected track was not found in the model catalogue."
    source_pos = int(positions[0])
    source = app_df.iloc[source_pos]
    source_genres = _genres(source)
    top_k = int(top_k)
    pool_size = min(len(app_df), 300 if model_type == "Hybrid" else top_k + 1)
    distances, indices = nn_model.kneighbors(X_app[source_pos].reshape(1, -1), n_neighbors=pool_size)
    results = []
    for distance, idx in zip(distances[0], indices[0]):
        idx = int(idx)
        if idx == source_pos:
            continue
        row = app_df.iloc[idx]
        candidate_genres = _genres(row)
        union = source_genres | candidate_genres
        intersection = source_genres & candidate_genres
        audio_similarity = float(1 - distance)
        genre_similarity = len(intersection) / len(union) if union else 0.0
        score = audio_similarity if model_type == "Audio-only" else audio_weight * audio_similarity + (1 - audio_weight) * genre_similarity
        results.append({
            "track_name": row["track_name"], "artists": row["artists"],
            "track_genre": row["track_genre"], "popularity": row["popularity"],
            "audio_similarity": audio_similarity, "genre_similarity": genre_similarity,
            "recommendation_score": score
        })
    result = pd.DataFrame(results).sort_values("recommendation_score", ascending=False).head(top_k).reset_index(drop=True)
    for col in ["audio_similarity", "genre_similarity", "recommendation_score"]:
        result[col] = result[col].round(4)
    message = (
        f"### Recommendations for {source['track_name']} — {source['artists']}\n\n"
        f"Model: **{model_type}** · Tracks: **{len(result)}**\n\n"
        "Scores indicate model similarity, not the probability that a listener will like a song."
    )
    return result, message

css = ".gradio-container {max-width: 1150px !important; margin: auto;} #title {text-align: center;}"
with gr.Blocks(theme=gr.themes.Soft(), css=css, title="SoundScout | Music Discovery") as demo:
    gr.Markdown("# SoundScout — Music Discovery", elem_id="title")
    gr.Markdown("Find similar tracks using audio features and genre-aware ranking.")
    with gr.Row():
        with gr.Column(scale=1):
            search_type = gr.Radio(["Song title", "Artist"], value="Artist", label="Search by")
            query = gr.Textbox(label="Search query", placeholder="e.g. Justin Bieber or Baby")
            search_btn = gr.Button("Search music", variant="primary")
            search_status = gr.Markdown()
        with gr.Column(scale=2):
            search_table = gr.Dataframe(label="Search results", interactive=False)
            selected_track = gr.Dropdown(label="Select a track", choices=[], interactive=True)
    gr.Markdown("---")
    gr.Markdown("## Recommendation settings")
    with gr.Row():
        model_choice = gr.Radio(["Hybrid", "Audio-only"], value="Hybrid", label="Model")
        top_k = gr.Slider(5, 20, value=10, step=1, label="Number of recommendations")
        audio_weight = gr.Slider(0.50, 1.00, value=0.75, step=0.05, label="Hybrid audio weight")
    recommend_btn = gr.Button("Recommend songs", variant="primary")
    recommendation_status = gr.Markdown()
    recommendation_table = gr.Dataframe(label="Recommendations", interactive=False)
    search_btn.click(search_music, inputs=[search_type, query], outputs=[selected_track, search_table, search_status])
    query.submit(search_music, inputs=[search_type, query], outputs=[selected_track, search_table, search_status])
    recommend_btn.click(recommend_music, inputs=[selected_track, model_choice, top_k, audio_weight], outputs=[recommendation_table, recommendation_status])

demo.launch(share=True, debug=True)